## Three Parts of this lab

### Part1: A simple "Agent" and "Agent Loop"
- Basically an LLM Call. We'll add tracing and streaming to the mix

### Part2: Adding Tool

A familiar one

### Part3: Adding Memory

So that different agent calls know about each other

In [12]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent 
from openai import OpenAI, AsyncOpenAI


from agents import Agent, Runner, trace, function_tool, SQLiteSession, set_default_openai_client, set_default_openai_api, set_trace_processors
from langsmith.integrations.openai_agents_sdk import OpenAIAgentsTracingProcessor

load_dotenv(override=True)

True

In [2]:
#set_tracing_disabled(True)

### Sidenote

The actual name of this framework on the official Python index pipi.org os openai-agents

So for your own projects in the future, you would do:
- pip install openai-agents or  uv add openai-agents
- followed by: from agents import Agent, Runner, trace

- Beware that doing a pip install agents would install something completely different - an older reinforcement learning library.

In [13]:
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")
ollama = AsyncOpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Tell SDK to use ollama client
set_default_openai_client(ollama)

# Tell the SDK to use chat completions
set_default_openai_api("chat_completions")



In [15]:
# Make an agent with name, instructions and model

agent = Agent(
    name="agent",
    instructions="You are a helpful assistant",
    model="qwen2.5:7b",
)

In [16]:
set_trace_processors([OpenAIAgentsTracingProcessor()])

In [17]:
result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

In [6]:
result.final_output

"Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to cross the road?\n\nBecause it already had a neural net for every possible scenario, and walking on roads wasn't part of its training data!"

In [7]:
# The details of the llm calls
result.to_input_list()

[{'content': 'Tell a joke about Autonomous AI Agents', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': "Sure! Here's a joke about autonomous AI agents:\n\nWhy did the autonomous AI agent refuse to cross the road?\n\nBecause it already had a neural net for every possible scenario, and walking on roads wasn't part of its training data!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'qwen2.5:7b', 'response_id': 'chatcmpl-18'}}]

### Adding Observability with a trace


In [9]:
## Trace works only with openai API Key. With Ollama it silently does nothing. Let us use langsmith
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")

print(result.final_output)


[non-fatal] Tracing client error 401. Response data is redacted.


Sure, here's a joke about Autonomous AI Agents:

Why did the Autonomous AI Agent break up with the Smartwatch?

Because it wanted to live a life not dictated by notifications!


[non-fatal] Tracing client error 401. Response data is redacted.


## Adding Tools




In [19]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = os.getenv("PUSHOVER_URL")
if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

if pushover_url:
    print("Pushover URL found and looks good")
else:
    print("Pushover URL not found")




Pushover user found and looks good
Pushover token found and looks good
Pushover URL found and looks good


In [20]:

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [21]:


push("HEY!!")



Push: HEY!!


In [22]:
@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [23]:
push_tool

FunctionTool(name='push_tool', description='Send the given message to the user as a push notification', params_json_schema={'properties': {'message': {'title': 'Message', 'type': 'string'}}, 'required': ['message'], 'title': 'push_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001EAA8F7B9B0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [24]:
push_tool.description

'Send the given message to the user as a push notification'